# Data Description

- Author: RUEI-HSIANG, CHANG（張睿庠；B12303030；AI協助）
- Date first created: 20261104
- Background: 既有候選人G1–G6核對及補查；蔣萬安G5=0由第9屆立委資料支持。
- Unit of observation: 主表及G5稽核每候選人1列；欄位證據每人6列；親屬表每候選人×親屬一筆觀察；缺口每候選人×缺欄一列；42筆線索另表。
- Sample period: 2026名冊及過去公職；checked_date為2026-11-04，檔名20261103為本輪起始日期。
- Known issues: 名冊沿用1,583筆登記人（議員1,502、縣市首長81），不是最終資格名單；未作樣本刪除。官方歷史身分連結、維基、媒體及本人陳述分層保留，不能宣稱全部官方認證。G5=1的35人是有明確報導支持，不等於完整窮盡終生選舉檔案。G6僅記有證據的正值，未知不得作無政治家族。原current三表保留先前版本，新G1–G6表為本輪入口；舊分析未重跑。 曾威、黃啟豪生日衝突保留原值並掛旗；張光儀歷史／當屆性別差異另掛旗。親屬表含超出親等與公職待查列，不能全部當G6=1。
- Definition for each variable: G1＝2026減出生年，非投票日足歲；G2＝來源記載性別；G3＝最高已完成學歷之三級分類（研究所／大學／大學以下），肄業、在學不直接認定完成；G4＝登記推薦政黨，無推薦不代表已確認無黨籍；G5＝首次參選任何民選公職（1首次、0曾參選、空白未知）；G6＝配偶或三親等內血親／姻親有公職來源記載。公職含民選與非民選，另記職務種類。G6空白不是0。堂表兄弟姊妹四親等排除。 candidate_key為名冊識別鍵；*_source定位來源；*_tier/status表示證據層級；checked_date本次核對日期；source_conflict_flag為來源差異。其餘欄位實際schema見下方各表輸出。
- Data path / SHA-256: 下表；另有驗證JSON `data/processed/candidate_g1_g6_verification_20261103.json`，SHA-256 `4ee227b612b2fafc4e09d876e3e945c16e9ca708cdb9b0886f4dc3ef8252c145`。
- Source data: `data/code/g1_g6_inputs_20261103.json`逐檔固定清單；原始說明`data/raw/g1_g6_20261103_documentation.ipynb`。
- Processing code: `data/code/g1_g6_20261103.ipynb`及同資料夾pipeline/helpers與人工核對JSON。
- Role: collection/evidence；後續analysis須處理未知與來源層級。
- Verification status: 儲存後重新讀取、列欄數、唯一鍵、分類語意與SHA-256核對。
- Downstream analysis input: `data/processed/candidates_g1_g6_current.csv`；證據表供追查。

| Data path | Rows × columns | SHA-256 |
| --- | --- | --- |
| `data/processed/candidates_g1_g6_current.csv` | 1583 × 36 | `d8695b306bd7eadbf0de5c8bb2f368c717d23d4ef7ae55cf3cf4b8fa1afbb59c` |
| `data/processed/candidate_g5_audit_20261103.csv` | 1583 × 13 | `6c361541ab417083bd0313225a6a382e0a94d4ad228fcba157bd23dee914e430` |
| `data/processed/candidate_g6_family_evidence_20261103.csv` | 156 × 19 | `04ecb38375192e2e83a16c7004e8af8c752b6658bb345de02c26f1b931dd8784` |
| `data/processed/candidate_g1_g6_field_evidence_20261103.csv` | 9498 × 7 | `3da3926cf1e03bb3bed3b48b3ce12562bc74f63d8c039deec8080f4f904f6b77` |
| `data/processed/candidate_g1_g6_source_conflicts_20261103.csv` | 2 × 7 | `7a99670f267f32c7f07544a8bc644aecd70b0787c689a739eaa89696ff72a08d` |
| `data/processed/candidate_g1_g6_identity_pending_20261103.csv` | 121 × 6 | `7cc0306385e10866543e1c8dae803e6daec7ac60026936a40984311e2dcc75c6` |
| `data/processed/candidate_g1_g6_remaining_worklist_20261103.csv` | 3669 × 7 | `927ce54e2ca4d8d53cba4bbf4b668d109ae88f107b046c10839962b856107724` |
| `data/processed/candidate_g5_first_time_lead_review_20261104.csv` | 42 × 10 | `69471d1da0425922543fd40dd09ca63cc4bd92722cd5d8aba7f29866d8afc0b0` |


In [1]:
from pathlib import Path
import hashlib, json, csv, sys
import pandas as pd
from IPython.display import display
R=next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def check(path, expected):
    p=(R/path).resolve()
    assert p.is_relative_to(R)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==expected, path
    return p


## Change for Your Own Data
以下固定指紋是本次已檢查版本，不以最新修改時間選檔。

In [2]:
EXPECTED={'data/processed/candidates_g1_g6_current.csv': 'd8695b306bd7eadbf0de5c8bb2f368c717d23d4ef7ae55cf3cf4b8fa1afbb59c', 'data/processed/candidate_g5_audit_20261103.csv': '6c361541ab417083bd0313225a6a382e0a94d4ad228fcba157bd23dee914e430', 'data/processed/candidate_g6_family_evidence_20261103.csv': '04ecb38375192e2e83a16c7004e8af8c752b6658bb345de02c26f1b931dd8784', 'data/processed/candidate_g1_g6_field_evidence_20261103.csv': '3da3926cf1e03bb3bed3b48b3ce12562bc74f63d8c039deec8080f4f904f6b77', 'data/processed/candidate_g1_g6_source_conflicts_20261103.csv': '7a99670f267f32c7f07544a8bc644aecd70b0787c689a739eaa89696ff72a08d', 'data/processed/candidate_g1_g6_identity_pending_20261103.csv': '7cc0306385e10866543e1c8dae803e6daec7ac60026936a40984311e2dcc75c6', 'data/processed/candidate_g1_g6_remaining_worklist_20261103.csv': '927ce54e2ca4d8d53cba4bbf4b668d109ae88f107b046c10839962b856107724', 'data/processed/candidate_g5_first_time_lead_review_20261104.csv': '69471d1da0425922543fd40dd09ca63cc4bd92722cd5d8aba7f29866d8afc0b0', 'data/processed/candidate_g1_g6_verification_20261103.json': '4ee227b612b2fafc4e09d876e3e945c16e9ca708cdb9b0886f4dc3ef8252c145'}
frames={}
for p,h in EXPECTED.items():
    f=check(p,h)
    if p.endswith('.csv'): frames[p]=pd.read_csv(f,dtype=str,keep_default_na=False)
main=frames['data/processed/candidates_g1_g6_current.csv']
summary=json.loads((R/'data/processed/candidate_g1_g6_verification_20261103.json').read_text())

## Summary

### Sample

In [3]:
for p,f in frames.items():
    print(p,f.shape)
    display(f.head(10))

data/processed/candidates_g1_g6_current.csv (1583, 36)


,candidate_key,candidate_name,county_city,office_type,electoral_district,G1_age_2026_minus_birth_year,birth_date,birth_year,G1_source_tier,G1_source,...,G5_evidence,G5_source,G5_audit_status,G6_political_family,G6_label,G6_evidence,G6_source,G6_evidence_tier,source_conflict_flag,checked_date
0,2026-C1-001,郭璽,臺北市,直轄市長,,68,1958-08-31,1958,prior_evidence_retained,https://db.cec.gov.tw/,...,2024總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/114_elcand.csv:28,official_historical_record_located_identity_su...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
1,2026-C1-002,沈伯洋,臺北市,直轄市長,,44,1982-06-07,1982,prior_evidence_retained,https://web.cec.gov.tw/api/file/9cdda387-4bb9-...,...,第11屆不分區立法委員（2024），屬曾參選任何民選公職；並非首次選臺北市長變數,data/raw/candidate_profiles_20261017/46775.htm...,official_profile_saved_text_rechecked,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
2,2026-C1-003,蕭文乾,臺北市,直轄市長,,52,1974-09-27,1974,prior_evidence_retained,https://web.cec.gov.tw/api/file/9cdda387-4bb9-...,...,2024不分區立委選舉公報候選人；非僅開政治獻金專戶,data/raw/cec_2024_party_list_bulletin.pages.js...,official_bulletin_saved_text_rechecked,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
3,2026-C1-004,唐新民,臺北市,直轄市長,,76,1950-12-25,1950,prior_evidence_retained,https://db.cec.gov.tw/,...,2022-111年地方公職人員選舉/C1/prv/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/100_elcand.csv:10,official_historical_record_located_identity_su...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
4,2026-C1-005,蔣萬安,臺北市,直轄市長,,48,1978-12-26,1978,prior_evidence_retained,https://db.cec.gov.tw/,...,立法院第9屆委員資料：2016/02/01至2020/01/31，臺北市第3選舉區；0=非首...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=1902,official_profile_rechecked_this_run,1,有三親等內或配偶公職來源記載,蔣孝嚴（父母）：第5－7屆中華民國立法委員；就任 2008年2月1日,https://zh.wikipedia.org/w/index.php?oldid=953...,published_secondary_relationships_see_family_t...,0,2026-11-04
5,2026-C1-006,林志成,臺北市,直轄市長,,61,1965-05-28,1965,prior_evidence_retained,https://db.cec.gov.tw/,...,2024總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/114_elcand.csv:209,official_historical_record_located_identity_su...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
6,2026-C1-007,蘇巧慧,新北市,直轄市長,,50,1976-04-05,1976,prior_evidence_retained,https://db.cec.gov.tw/,...,2020總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/095_elcand.csv:66,official_historical_record_located_identity_su...,1,有三親等內或配偶公職來源記載,蘇貞昌（父母）：第11任臺北縣縣長；就任 1997年12月20日 / 第3屆中華民國立法委員...,https://zh.wikipedia.org/w/index.php?oldid=952...,published_secondary_relationships_see_family_t...,0,2026-11-04
7,2026-C1-008,李四川,新北市,直轄市長,,68,1958-02-01,1958,media_2026_explicit_birth_date,https://www.cna.com.tw/news/aipl/202610275002....,...,026.08.20 07:30 最後更新時間： 2026.08.20 07:30 李四川今透...,https://news.tvbs.com.tw/politics/4009743,explicit_first_time_report_not_exhaustive_arch...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
8,2026-C1-009,蘇輝湟,新北市,直轄市長,,50,1976-11-10,1976,prior_evidence_retained,https://db.cec.gov.tw/,...,2022-111年地方公職人員選舉/V1/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/112_elcand.csv:9805,official_historical_record_located_identity_su...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04
9,2026-C1-010,張善政,桃園市,直轄市長,,72,1954-06-24,1954,prior_evidence_retained,https://db.cec.gov.tw/,...,2020總統立委/總統/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/098_elcand.csv:4,official_historical_record_located_identity_su...,,未知／尚無足夠來源；不是0,,,,0,2026-11-04


data/processed/candidate_g5_audit_20261103.csv (1583, 13)


,candidate_key,candidate_name,office_type,previous_value,audited_value,meaning,prior_election_year_example,evidence,source,audit_status,changed,note,checked_date
0,2026-C1-001,郭璽,直轄市長,0,0,非首次參選任何民選公職,2024,2024總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/114_elcand.csv:28,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
1,2026-C1-002,沈伯洋,直轄市長,0,0,非首次參選任何民選公職,2024,第11屆不分區立法委員（2024），屬曾參選任何民選公職；並非首次選臺北市長變數,data/raw/candidate_profiles_20261017/46775.htm...,official_profile_saved_text_rechecked,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
2,2026-C1-003,蕭文乾,直轄市長,0,0,非首次參選任何民選公職,2024,2024不分區立委選舉公報候選人；非僅開政治獻金專戶,data/raw/cec_2024_party_list_bulletin.pages.js...,official_bulletin_saved_text_rechecked,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
3,2026-C1-004,唐新民,直轄市長,0,0,非首次參選任何民選公職,2022,2022-111年地方公職人員選舉/C1/prv/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/100_elcand.csv:10,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
4,2026-C1-005,蔣萬安,直轄市長,0,0,非首次參選任何民選公職,2016,立法院第9屆委員資料：2016/02/01至2020/01/31，臺北市第3選舉區；0=非首...,https://www.ly.gov.tw/Pages/List.aspx?nodeid=1902,official_profile_rechecked_this_run,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
5,2026-C1-006,林志成,直轄市長,0,0,非首次參選任何民選公職,2024,2024總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/114_elcand.csv:209,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
6,2026-C1-007,蘇巧慧,直轄市長,0,0,非首次參選任何民選公職,2020,2020總統立委/區域立委/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/095_elcand.csv:66,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
7,2026-C1-008,李四川,直轄市長,,1,首次參選任何民選公職,,026.08.20 07:30 最後更新時間： 2026.08.20 07:30 李四川今透...,https://news.tvbs.com.tw/politics/4009743,explicit_first_time_report_not_exhaustive_arch...,1,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
8,2026-C1-009,蘇輝湟,直轄市長,0,0,非首次參選任何民選公職,2022,2022-111年地方公職人員選舉/V1/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/112_elcand.csv:9805,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04
9,2026-C1-010,張善政,直轄市長,0,0,非首次參選任何民選公職,2020,2020總統立委/總統/elcand.csv；既有生日佐證身分連結,data/raw/cec_all_offices/098_elcand.csv:4,official_historical_record_located_identity_su...,0,例示年份不是終生最早參選年份；首次選本職與首次參選任何公職不同。,2026-11-04


data/processed/candidate_g6_family_evidence_20261103.csv (156, 19)


,candidate_key,candidate_name,relative_name,relationship,degree,kinship_type,in_scope,relative_public_office,office_type,relationship_text,relationship_source_url,relationship_source_file,office_source_url,office_source_file,source_tier,review_status,checked_date,corroborating_source_url,corroborating_source_file
0,2026-C1-005,蔣萬安,蔣孝嚴,父母,1,血親,1,第5－7屆中華民國立法委員；就任 2008年2月1日,民選職務（來源記載）,父：蔣孝嚴\n母：黃美倫,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_08.json,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_rel_02.json,secondary_with_separate_news_corroboration,secondary_relationship_and_public_office,2026-11-04,https://www.cna.com.tw/news/aipl/201711010159....,data/raw/g1_g6_20261103/chiang_family.html
1,2026-C1-005,蔣萬安,石舫亘,配偶,,配偶,1,,,{{marriage|石舫亘|2008}},https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_08.json,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_rel_02.json,secondary_wikipedia,relative_office_or_relation_pending,2026-11-04,,
2,2026-C1-007,蘇巧慧,蘇貞昌,父母,1,血親,1,第11任臺北縣縣長；就任 1997年12月20日 / 第3屆中華民國立法委員；就任 1996...,民選職務（來源記載）,蘇貞昌（父）\n詹秀齡（母）,https://zh.wikipedia.org/w/index.php?oldid=952...,data/raw/g1_g6_20261103/wiki_batch_08.json,https://zh.wikipedia.org/w/index.php?oldid=944...,data/raw/g1_g6_20261103/wiki_batch_rel_02.json,secondary_wikipedia,secondary_relationship_and_public_office,2026-11-04,,
3,2026-C1-012,何欣純,林昶佐,堂表親（超出三親等）,4,血親,0,第9－10屆中華民國立法委員；就任 2016年2月1日,民選職務（來源記載）,林昶佐（表弟）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_00.json,https://zh.wikipedia.org/w/index.php?oldid=950...,data/raw/g1_g6_20261103/wiki_batch_rel_01.json,secondary_wikipedia,outside_three_degrees,2026-11-04,,
4,2026-C1-013,江啟臣,江春男,堂表親（超出三親等）,5,血親,0,,,江海泉（父、2025年歿）\n江春男（堂叔）\n劉盛良（岳父）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_06.json,https://zh.wikipedia.org/w/index.php?oldid=947...,data/raw/g1_g6_20261103/wiki_batch_rel_01.json,secondary_wikipedia,outside_three_degrees,2026-11-04,,
5,2026-C1-013,江啟臣,劉盛良,配偶父母,1,姻親,1,{{ROC}}第3－4、6－7屆立法委員；就任 2008年2月1日,民選職務（來源記載）,江海泉（父、2025年歿）\n江春男（堂叔）\n劉盛良（岳父）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_06.json,https://zh.wikipedia.org/w/index.php?oldid=943...,data/raw/g1_g6_20261103/wiki_batch_rel_00.json,secondary_wikipedia,secondary_relationship_and_public_office,2026-11-04,,
6,2026-C1-015,謝龍介,謝克洋,三等旁系親屬,3,血親,1,,,謝岩介（弟）\n謝克洋（姪子）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_09.json,,,secondary_wikipedia,relative_office_or_relation_pending,2026-11-04,,
7,2026-C1-018,陳亭妃,陳怡珍,兄弟姊妹,2,血親,1,第1－4屆臺南市議員；就任 2010年12月25日,民選職務（來源記載）,陳怡珍（妹）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_10.json,https://zh.wikipedia.org/w/index.php?oldid=943...,data/raw/g1_g6_20261103/wiki_batch_rel_03.json,secondary_wikipedia,secondary_relationship_and_public_office,2026-11-04,,
8,2026-C1-022,柯志恩,柯文福,父母,1,血親,1,{{PIN}}第5任縣長；就任 1973年2月1日,民選職務（來源記載）,柯文福（父）,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_05.json,https://zh.wikipedia.org/w/index.php?oldid=939...,data/raw/g1_g6_20261103/wiki_batch_rel_01.json,secondary_with_separate_news_corroboration,secondary_relationship_and_public_office,2026-11-04,https://news.tvbs.com.tw/politics/1871102,data/raw/g1_g6_20261103/ko_family.html
9,2026-C2-024,鄭朝方,鄭永金,父母,1,血親,1,第9任新竹縣縣長；就任 2001年12月20日 / 第3－4屆中華民國立法委員；就任 199...,民選職務（來源記載）,鄭永金,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_09.json,https://zh.wikipedia.org/w/index.php?oldid=953...,data/raw/g1_g6_20261103/wiki_batch_rel_02.json,secondary_wikipedia,secondary_relationship_and_public_office,2026-11-04,,


... (6 outputs left out)

data/processed/candidate_g1_g6_remaining_worklist_20261103.csv (3669, 7)


,candidate_key,candidate_name,county_city,office_type,field,status,next_step
0,2026-C1-001,郭璽,臺北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
1,2026-C1-002,沈伯洋,臺北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
2,2026-C1-003,蕭文乾,臺北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
3,2026-C1-004,唐新民,臺北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
4,2026-C1-006,林志成,臺北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
5,2026-C1-008,李四川,新北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
6,2026-C1-009,蘇輝湟,新北市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
7,2026-C1-010,張善政,桃園市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
8,2026-C1-011,黃世杰,桃園市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0
9,2026-C1-014,洪麗華,臺中市,直轄市長,G6,unknown_not_zero,三親等／配偶關係與公職證據；不得用查無結果填0


data/processed/candidate_g5_first_time_lead_review_20261104.csv (42, 10)


,candidate_key,candidate_name,county_city,source_url,source_file,assert_phrase,source_excerpt,decision,note,checked_date
0,2026-C1-008,李四川,新北市,https://news.tvbs.com.tw/politics/4009743,data/raw/g1_g6_20261103/first_time_f0d67d34c94...,以政壇老將之姿首次參選,026.08.20 07:30 最後更新時間： 2026.08.20 07:30 李四川今透...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。 歷史同名李四川生日1960-0...,2026-11-04
1,2026-C2-046,林國漳,宜蘭縣,https://udn.com/vote2026/story/124652/9521613,data/raw/g1_g6_20261103/first_time_fb479b4d281...,從來沒有選戰經驗的林國漳,前縣長 游錫堃 邀請他擔任縣選委會選監小組召集人，返鄉後原以為律師與老師將是一生志業，未料去...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
2,2026-C2-055,王志偉,臺東縣,https://udn.com/vote2026/story/124652/9734975,data/raw/g1_g6_20261103/first_time_8099e6fbb6f...,過去從未參選村里長或民意代表,認為應該滿足的現況。他希望透過參選，鼓勵縣民投下理性的一票，為台東未來尋找新的可能。 王志偉...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。 原文明確未參選村里長或民意代表...,2026-11-04
3,2026|直轄市議員|臺北市第1選舉區|賴苡任,賴苡任,臺北市,https://news.nextapple.com/politics/20260401/A...,data/raw/g1_g6_20261103/first_time_e90bdc25de7...,首次參選的他,合、審查沒問題，但一個人的不同意就足以讓他沒資格參選，氣得他質疑：「陳啟宏委員，聽說你跟黃子...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
4,2026|直轄市議員|臺北市第3選舉區|許甫,許甫,臺北市,https://udn.com/news/amp/story/124652/9283631,data/raw/g1_g6_20261103/first_time_5906b0303b3...,自己是第一次參選,辦活動適當時機會邀請，但不會打選情告急牌。 民眾黨中正萬華市議員參選人吳怡萱分享近期選戰節奏...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
5,2026|直轄市議員|臺北市第5選舉區|陳力新,陳力新,臺北市,https://www.lishin.taipei/,data/raw/g1_g6_20261103/first_time_0acb3c8db17...,,,pending_no_explicit_claim_or_fetch_failed,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
6,2026|直轄市議員|臺北市第6選舉區|劉品妡,劉品妡,臺北市,https://www.bo6s.com.tw/news_detail.php?NewsID...,data/raw/g1_g6_20261103/first_time_b1f8d7cbadf...,雖是首次參選,17:22 瀏覽: 3323 波新聞─范政達／台北市 台北市大安文山區民進黨市議員參選人劉...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
7,2026|直轄市議員|新北市第3選舉區|陳岱吟,陳岱吟,新北市,https://video.udn.com/news/1318889,data/raw/g1_g6_20261103/first_time_eb9f5a1be1a...,首次參選的新莊區新人陳岱吟,/ 10 12:19 「新新板土」連線完成民進黨內初選登記 林右昌陪同站台 民進黨新北市議...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
8,2026|直轄市議員|新北市第8選舉區|吳昇翰,吳昇翰,新北市,https://tw.news.yahoo.com/%E6%B0%91%E9%80%B2%E...,data/raw/g1_g6_20261103/first_time_eddf7fdde4a...,首次參選的吳昇翰,結果，競爭激烈的第八選區（土城、樹林、三峽、鶯歌），新人 吳昇翰 順利通過初選，取得提名門票...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04
9,2026|直轄市議員|桃園市第3選舉區|林晏良,林晏良,桃園市,https://www.taisounds.com/news/content/70/236723,data/raw/g1_g6_20261103/first_time_92972e88283...,這是他第一次參選,必移除」該廣告看板，「造成社會大眾觀感不佳的部分，本人在此誠摯的表達歉意；再次抱歉，未來一定...,accepted_explicit_first_time_report,依媒體或本人明確首次參選敘述；不等於已完整排除所有歷史紀錄。,2026-11-04


### Summary Stats

In [4]:
display(pd.DataFrame([{'field':k,'known':v,'unknown':1583-v} for k,v in summary['coverage'].items()]))
display(main.G5_label.value_counts().to_frame('count'))
for p,f in frames.items():
    print(p)
    display(pd.DataFrame({'dtype':f.dtypes.astype(str),'blank':f.eq('').sum()}))

,field,known,unknown
0,G1,857,726
1,G2,1582,1
2,G3,858,725
3,G4,1583,0
4,G5,856,727
5,G6,93,1490


,count
G5_label,
非首次參選任何民選公職,821
未知／未核實,727
首次參選任何民選公職,35


data/processed/candidates_g1_g6_current.csv


,dtype,blank
candidate_key,object,0
candidate_name,object,0
county_city,object,0
office_type,object,0
electoral_district,object,81
G1_age_2026_minus_birth_year,object,726
birth_date,object,730
birth_year,object,726
G1_source_tier,object,726
G1_source,object,729


data/processed/candidate_g5_audit_20261103.csv


,dtype,blank
candidate_key,object,0
candidate_name,object,0
office_type,object,0
previous_value,object,805
audited_value,object,727
meaning,object,0
prior_election_year_example,object,764
evidence,object,727
source,object,727
audit_status,object,0


... (8 outputs left out)

data/processed/candidate_g1_g6_remaining_worklist_20261103.csv


,dtype,blank
candidate_key,object,0
candidate_name,object,0
county_city,object,0
office_type,object,0
field,object,0
status,object,0
next_step,object,0


data/processed/candidate_g5_first_time_lead_review_20261104.csv


,dtype,blank
candidate_key,object,0
candidate_name,object,0
county_city,object,0
source_url,object,0
source_file,object,0
assert_phrase,object,7
source_excerpt,object,7
decision,object,0
note,object,0
checked_date,object,0


### Assertions

In [5]:
assert len(main)==1583 and main.candidate_key.is_unique
assert set(main.G5_first_time_any_elected_office)<={'','0','1'}
assert (main.G5_first_time_any_elected_office=='1').sum()==35
assert (main.G5_first_time_any_elected_office=='0').sum()==821
assert set(main.G6_political_family)<={'','1'}
assert main.loc[main.candidate_name=='蔣萬安','G5_first_time_any_elected_office'].tolist()==['0']
for p,v in summary['outputs'].items(): assert frames[p].shape==(v['rows'],v['columns'])
assert len(frames['data/processed/candidate_g1_g6_field_evidence_20261103.csv'])==1583*6
print('all saved-file and semantic checks passed')

all saved-file and semantic checks passed
